# Deep dive 2/13: `products`

More exhaustive than the `customers` notebook, per request -- `products` has more moving parts (a UNIQUE constraint, an FK to `customers` *and* one to `branches`, a currency conversion join against `silver.v_fx_latest_to_usd`, and fields that are only meaningful for a subset of `product_type` values), so there's more that can go quietly wrong without erroring.

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `products_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 400,000 rows, Source: Core Banking, Partition: monthly_snapshot.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 1. Column inventory -- Bronze vs. Silver vs. `products_spec`

`products_spec` adds two derived columns Bronze doesn't have at all: `current_balance_usd` and `credit_limit_usd` (computed via a join to `silver.v_fx_latest_to_usd`, not a straight cast). Confirming both the straight-through columns and these two derived ones land correctly.

In [2]:
expected_silver_columns = [
    "product_id", "customer_id", "product_type", "product_number", "currency",
    "current_balance", "current_balance_usd", "credit_limit", "credit_limit_usd",
    "interest_rate", "opening_date", "expiration_date", "opening_branch_id",
    "product_status", "opening_channel", "has_linked_app", "days_past_due",
    "last_transaction_date", "last_updated",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_products'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_products:", missing or "none")
print("Extra in silver.dim_products:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.dim_products: none
Extra in silver.dim_products: none

Column order as built: ['product_id', 'customer_id', 'product_type', 'product_number', 'currency', 'current_balance', 'current_balance_usd', 'credit_limit', 'credit_limit_usd', 'interest_rate', 'opening_date', 'expiration_date', 'opening_branch_id', 'product_status', 'opening_channel', 'has_linked_app', 'days_past_due', 'last_transaction_date', 'last_updated']


In [3]:
print("Bronze columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='products' ORDER BY ordinal_position
"""))

print("Silver columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_products' ORDER BY ordinal_position
"""))

Bronze columns:


,column_name,data_type
0,product_id,VARCHAR
1,customer_id,VARCHAR
2,product_type,VARCHAR
3,product_number,VARCHAR
4,currency,VARCHAR
5,current_balance,VARCHAR
6,credit_limit,VARCHAR
7,interest_rate,VARCHAR
8,opening_date,VARCHAR
9,expiration_date,VARCHAR


Silver columns:


,column_name,data_type
0,product_id,VARCHAR
1,customer_id,VARCHAR
2,product_type,VARCHAR
3,product_number,VARCHAR
4,currency,VARCHAR
5,current_balance,DOUBLE
6,current_balance_usd,DOUBLE
7,credit_limit,DOUBLE
8,credit_limit_usd,DOUBLE
9,interest_rate,DOUBLE


## 2. Row-count reconciliation -- including a fan-out check

Same dedup invariant as `customers`, but `products_spec` also does a `LEFT JOIN` to `silver.v_fx_latest_to_usd` to compute the USD columns. If that view ever has more than one row per `source_currency`, the join fans out and Silver would end up with **more** rows than Bronze has distinct `product_id` -- a bug that row-count alone can catch even without inspecting the FX logic directly (section 7 does that part).

In [4]:
bronze_total = q("SELECT count(*) AS n FROM bronze.products")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT product_id) AS n FROM bronze.products")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_products")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT product_id) AS n FROM silver.dim_products")["n"][0]

print(f"bronze.products:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct product_id")
print(f"silver.dim_products:  {silver_total:,} rows, {silver_distinct_pk:,} distinct product_id")
print()
print("Duplicate raw rows in Bronze (resends/reloads):", bronze_total - bronze_distinct_pk)
if silver_total == bronze_distinct_pk:
    print("OK -- row count matches, no FX-join fan-out")
elif silver_total > bronze_distinct_pk:
    print("FAN-OUT -- silver has MORE rows than distinct product_id, the FX join is duplicating rows")
else:
    print("MISMATCH -- silver has FEWER rows than distinct product_id, investigate")

bronze.products:      400,000 rows, 400,000 distinct product_id
silver.dim_products:  400,000 rows, 400,000 distinct product_id

Duplicate raw rows in Bronze (resends/reloads): 0
OK -- row count matches, no FX-join fan-out


In [5]:
# Any duplicate product_id left in Silver itself? Should always be empty.
q("SELECT product_id, count(*) AS n FROM silver.dim_products GROUP BY 1 HAVING count(*) > 1")

,product_id,n


In [6]:
# If Bronze has any duplicate product_id, confirm Silver kept the one with the latest _ingested_at
# (dedup_order_by="p._ingested_at DESC" in products_spec). Empty result if Bronze has no dupes.
q("""
    WITH dupes AS (
        SELECT product_id FROM bronze.products GROUP BY 1 HAVING count(*) > 1
    ), ranked AS (
        SELECT b.product_id, b._ingested_at,
               row_number() OVER (PARTITION BY b.product_id ORDER BY b._ingested_at DESC) AS rn
        FROM bronze.products b JOIN dupes d USING (product_id)
    )
    SELECT r.product_id, r._ingested_at AS latest_bronze_ingested_at, s.last_updated
    FROM ranked r JOIN silver.dim_products s ON s.product_id = r.product_id
    WHERE r.rn = 1
    LIMIT 20
""")

,product_id,latest_bronze_ingested_at,last_updated


## 3. NOT NULL audit

Dictionary NOT NULL columns: `product_id`, `customer_id`, `product_type`, `product_number`, `currency`, `current_balance`, `opening_date`, `opening_branch_id`, `product_status`, `opening_channel`, `has_linked_app`, `last_updated`. `credit_limit`, `interest_rate`, `expiration_date`, `days_past_due`, `last_transaction_date` are legitimately nullable (credit-specific or optional fields) -- see section 10 for whether their null rate lines up with the product mix, not just whether they're populated.

In [7]:
declared_not_null = [
    "product_id", "customer_id", "product_type", "product_number", "currency",
    "current_balance", "opening_date", "opening_branch_id", "product_status",
    "opening_channel", "has_linked_app", "last_updated",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.products")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.products
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

pd.DataFrame(rows)

,column,declared,actual_nulls,null_rate
0,product_id,NOT NULL,0,0.0
1,customer_id,NOT NULL,0,0.0
2,product_type,NOT NULL,0,0.0
3,product_number,NOT NULL,0,0.0
4,currency,NOT NULL,0,0.0
5,current_balance,NOT NULL,0,0.0
6,opening_date,NOT NULL,0,0.0
7,opening_branch_id,NOT NULL,0,0.0
8,product_status,NOT NULL,0,0.0
9,opening_channel,NOT NULL,0,0.0


## 4. UNIQUE constraint audit -- `product_number`

Declared `NOT NULL, UNIQUE` in the dictionary. Silver carries it through untouched (no transform), so any Bronze-level duplication passes straight into Silver.

In [8]:
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT product_number) AS distinct_product_number,
        count(*) FILTER (WHERE product_number IS NULL OR trim(product_number) = '') AS nulls
    FROM bronze.products
""")

,total,distinct_product_number,nulls
0,400000,399994,0


In [9]:
# Which product_number values repeat, and is it the same product_id each time (a resend) or two
# genuinely different products sharing a number (a real uniqueness violation)?
q("""
    SELECT product_number, count(*) AS n, count(DISTINCT product_id) AS distinct_product_ids
    FROM bronze.products
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
    LIMIT 20
""")

,product_number,n,distinct_product_ids
0,LOAN-37068248,2,2
1,4929931697,2,2
2,LOAN-64525937,2,2
3,LOAN-90296514,2,2
4,7341595333,2,2
5,LOAN-67882840,2,2


## 5. Domain checks

`product_status` is the only one of these `contracts.py` currently gates (`{Active, Closed, Blocked, Suspended}` -- matches the dictionary here, unlike `customers.customer_status`). `product_type`, `currency`, and `opening_channel` aren't gated at all -- worth seeing what's actually in them, especially since the dictionary's `product_type` cell was truncated in the PDF table and can't be trusted as a complete list.

In [10]:
# product_type -- dictionary text was cut off mid-list in the PDF table, so treat this query,
# not the dictionary, as the authoritative list.
q("SELECT product_type, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

,product_type,n
0,Cuenta Ahorro,120203
1,Tarjeta Crédito,100102
2,Cuenta Corriente,99979
3,Tarjeta Débito,39938
4,Préstamo Personal,19960
5,Préstamo Hipotecario,11910
6,Inversión,5859
7,Seguro,2049


In [11]:
# currency -- dictionary: MXN, COP, ARS, USD
q("SELECT currency, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

,currency,n
0,USD,220501
1,COP,107975
2,ARS,71524


In [12]:
# product_status -- dictionary + contracts.py both declare: Active, Blocked, Closed, Suspended
q("SELECT product_status, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

,product_status,n
0,Active,339965
1,Closed,32039
2,Blocked,19935
3,Suspended,8061


In [13]:
# opening_channel -- dictionary: Branch, Web, App, Call Center
q("SELECT opening_channel, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

,opening_channel,n
0,Branch,199838
1,Web,100282
2,App,79726
3,Call Center,20154


## 6. Foreign key integrity -- reproduced directly, not just trusted from the quality report

`customer_id -> customers.customer_id` and `opening_branch_id -> branches.branch_id` are both declared in `contracts.py`. The branch one was already found clean (0 orphans) during the branch_id investigation -- re-running it here directly against Silver, alongside the customer one, so this notebook doesn't have to take that on faith.

In [14]:
q("""
    SELECT
        count(*) AS total_products,
        count(*) FILTER (WHERE customer_id IS NOT NULL) AS customer_id_populated,
        count(*) FILTER (
            WHERE customer_id IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM silver.dim_customers c WHERE c.customer_id = p.customer_id)
        ) AS customer_id_orphans
    FROM silver.dim_products p
""")

,total_products,customer_id_populated,customer_id_orphans
0,400000,400000,0


In [15]:
q("""
    SELECT
        count(*) AS total_products,
        count(*) FILTER (WHERE opening_branch_id IS NOT NULL) AS branch_id_populated,
        count(*) FILTER (
            WHERE opening_branch_id IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM silver.dim_branches b WHERE b.branch_id = p.opening_branch_id)
        ) AS branch_id_orphans
    FROM silver.dim_products p
""")

,total_products,branch_id_populated,branch_id_orphans
0,400000,400000,0


## 7. Currency conversion correctness -- the part that isn't a simple cast

`current_balance_usd`/`credit_limit_usd` come from `TRY_CAST(...) * CASE WHEN currency='USD' THEN 1 ELSE fx.rate_to_usd END`, joined against `silver.v_fx_latest_to_usd`. Three ways this can go quietly wrong without erroring: (a) a currency in `products` that `v_fx_latest_to_usd` doesn't cover, silently producing NULL instead of a converted amount; (b) the USD passthrough not being exact; (c) a non-USD conversion using the wrong rate.

In [16]:
# (a) FX coverage -- every currency used in products should have a row in v_fx_latest_to_usd.
print("Currencies in products:")
display(q("SELECT DISTINCT currency FROM bronze.products ORDER BY 1"))

print("Currencies covered by v_fx_latest_to_usd:")
display(q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency"))

Currencies in products:


,currency
0,ARS
1,COP
2,USD


Currencies covered by v_fx_latest_to_usd:


,source_currency,rate_to_usd,rate_date
0,ARS,0.002873,2026-06-17
1,COP,0.000248,2026-06-17
2,MXN,0.058641,2026-06-17


In [17]:
# Rows where current_balance is populated but current_balance_usd came out NULL -- the FX join
# silently failed for these. Should be 0 given full coverage above; if not, this is exactly the
# kind of gap that wouldn't show up as an error anywhere.
q("""
    SELECT currency, count(*) AS n
    FROM silver.dim_products
    WHERE current_balance IS NOT NULL AND current_balance_usd IS NULL
    GROUP BY 1
""")

,currency,n


In [18]:
# (b) USD passthrough -- for currency='USD' rows, current_balance_usd should equal current_balance
# exactly (rate forced to 1), not go through a real (and irrelevant) FX rate.
q("""
    SELECT count(*) AS mismatched_usd_passthrough
    FROM silver.dim_products
    WHERE currency = 'USD'
      AND current_balance IS NOT NULL
      AND current_balance_usd IS NOT NULL
      AND abs(current_balance - current_balance_usd) > 0.01
""")

,mismatched_usd_passthrough
0,0


In [19]:
# (c) Non-USD conversion -- spot-check a sample against an independent re-computation using the
# same v_fx_latest_to_usd view, to catch a join predicate or rounding bug.
q("""
    SELECT
        p.product_id, p.currency, p.current_balance, p.current_balance_usd,
        fx.rate_to_usd,
        round(p.current_balance * fx.rate_to_usd, 2) AS recomputed_usd,
        abs(p.current_balance_usd - round(p.current_balance * fx.rate_to_usd, 2)) AS diff
    FROM silver.dim_products p
    LEFT JOIN silver.v_fx_latest_to_usd fx ON fx.source_currency = p.currency
    WHERE p.currency <> 'USD' AND p.current_balance IS NOT NULL
    ORDER BY diff DESC
    LIMIT 10
""")

,product_id,currency,current_balance,current_balance_usd,rate_to_usd,recomputed_usd,diff
0,PRD-ZTYK6X51U8QQ,COP,7847036.29,1946.065,0.000248,1946.06,0.005
1,PRD-7PBT53PL05YS,ARS,764387.40,2196.085,0.002873,2196.09,0.005
2,PRD-0O0VC1DIPAYL,COP,27142358.87,6731.305,0.000248,6731.30,0.005
3,PRD-BOHI9G83BGJX,COP,14250020.16,3534.005,0.000248,3534.00,0.005
4,PRD-3J9NBO6YSDLC,COP,1781270.16,441.755,0.000248,441.75,0.005
5,PRD-J4BPX2UFLXM9,COP,9921229.84,2460.465,0.000248,2460.47,0.005
6,PRD-QOQ86TRXA6OV,COP,11004979.84,2729.235,0.000248,2729.24,0.005
7,PRD-QKC66XSOOZB0,COP,7661229.84,1899.985,0.000248,1899.99,0.005
8,PRD-I8IV4RBX7N29,ARS,4256072.05,12227.695,0.002873,12227.69,0.005
9,PRD-2CN4X9JL6TXD,COP,23900181.45,5927.245,0.000248,5927.24,0.005


In [20]:
# Same three checks, but for credit_limit_usd -- it's only populated for a subset of products
# (credit_limit itself is nullable), so NULL alone isn't a bug here; only a populated
# credit_limit with a NULL credit_limit_usd is.
q("""
    SELECT currency, count(*) AS n
    FROM silver.dim_products
    WHERE credit_limit IS NOT NULL AND credit_limit_usd IS NULL
    GROUP BY 1
""")

,currency,n


## 8. Numeric range sanity

In [21]:
q("""
    SELECT
        min(current_balance) AS min_balance, max(current_balance) AS max_balance,
        count(*) FILTER (WHERE current_balance < 0) AS negative_balances,
        count(*) FILTER (WHERE current_balance IS NULL) AS nulls
    FROM silver.dim_products
""")

,min_balance,max_balance,negative_balances,nulls
0,0.0,8.815115e+08,0,0


In [22]:
q("""
    SELECT
        min(credit_limit) AS min_credit_limit, max(credit_limit) AS max_credit_limit,
        count(*) FILTER (WHERE credit_limit < 0) AS negative_credit_limits,
        count(*) FILTER (WHERE credit_limit IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_products
""")

,min_credit_limit,max_credit_limit,negative_credit_limits,populated,total
0,1000.33,5.999842e+08,0,125317,400000


In [23]:
q("""
    SELECT
        min(interest_rate) AS min_rate, max(interest_rate) AS max_rate,
        count(*) FILTER (WHERE interest_rate < 0) AS negative_rates,
        count(*) FILTER (WHERE interest_rate IS NOT NULL) AS populated
    FROM silver.dim_products
""")

,min_rate,max_rate,negative_rates,populated
0,0.0,45.0,0,359934


In [24]:
q("""
    SELECT
        min(days_past_due) AS min_dpd, max(days_past_due) AS max_dpd,
        count(*) FILTER (WHERE days_past_due < 0) AS negative_dpd,
        count(*) FILTER (WHERE days_past_due IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_products
""")

,min_dpd,max_dpd,negative_dpd,populated,total
0,0.0,180.0,0,125350,400000


## 9. Date sanity

In [25]:
# opening_date vs. the dataset's overall June 2023 - June 2026 window (note: for customers this
# assumption didn't hold for registration_date -- check it fresh here rather than assume the same
# conclusion applies).
q("""
    SELECT
        min(opening_date) AS earliest_opening, max(opening_date) AS latest_opening,
        count(*) FILTER (WHERE opening_date < DATE '2023-06-17') AS before_window,
        count(*) FILTER (WHERE opening_date > DATE '2026-06-17') AS after_window
    FROM silver.dim_products
""")

,earliest_opening,latest_opening,before_window,after_window
0,2018-06-18,2026-06-17,250127,0


In [26]:
# expiration_date should be after opening_date when populated (it's "for term products" per the
# dictionary, so NULL is expected for most rows -- only checking internal consistency where set).
q("""
    SELECT
        count(*) FILTER (WHERE expiration_date IS NOT NULL) AS populated,
        count(*) FILTER (WHERE expiration_date IS NOT NULL AND expiration_date <= opening_date) AS expires_before_or_on_opening
    FROM silver.dim_products
""")

,populated,expires_before_or_on_opening
0,133161,0


In [27]:
# last_transaction_date should never precede opening_date, and last_updated should never precede
# either. Also checking for the same kind of "future" anomaly customers.last_updated had.
q("""
    SELECT
        max(last_updated) AS latest_last_updated,
        count(*) FILTER (WHERE last_transaction_date IS NOT NULL AND TRY_CAST(last_transaction_date AS DATE) < opening_date) AS txn_before_opening,
        count(*) FILTER (WHERE last_updated < opening_date) AS updated_before_opening,
        count(*) FILTER (WHERE last_updated > TIMESTAMP '2026-06-17 23:59:59') AS updated_after_dataset_window
    FROM silver.dim_products
""")

,latest_last_updated,txn_before_opening,updated_before_opening,updated_after_dataset_window
0,2027-06-15 02:26:58,0,0,25113


## 10. Cross-field consistency -- are credit-only fields actually credit-only?

`credit_limit` and `days_past_due` are documented as credit-product fields. This checks whether that's actually how the data is shaped: populated mostly for credit-type products, NULL for deposit-type ones (or not -- worth knowing either way, since nothing in Silver enforces this).

In [28]:
q("""
    SELECT
        product_type,
        count(*) AS n,
        count(*) FILTER (WHERE credit_limit IS NOT NULL) AS has_credit_limit,
        round(100.0 * count(*) FILTER (WHERE credit_limit IS NOT NULL) / count(*), 1) AS pct_has_credit_limit,
        count(*) FILTER (WHERE days_past_due IS NOT NULL) AS has_days_past_due,
        round(100.0 * count(*) FILTER (WHERE days_past_due IS NOT NULL) / count(*), 1) AS pct_has_dpd
    FROM silver.dim_products
    GROUP BY 1
    ORDER BY n DESC
""")

,product_type,n,has_credit_limit,pct_has_credit_limit,has_days_past_due,pct_has_dpd
0,Cuenta Ahorro,120203,0,0.0,0,0.0
1,Tarjeta Crédito,100102,95043,94.9,95033,94.9
2,Cuenta Corriente,99979,0,0.0,0,0.0
3,Tarjeta Débito,39938,0,0.0,0,0.0
4,Préstamo Personal,19960,18968,95.0,18993,95.2
5,Préstamo Hipotecario,11910,11306,94.9,11324,95.1
6,Inversión,5859,0,0.0,0,0.0
7,Seguro,2049,0,0.0,0,0.0


## 11. `has_linked_app` -- boolean parse check, same pattern as `customers.accepts_marketing`

In [29]:
print("Bronze raw values:")
display(q("SELECT typeof(has_linked_app), has_linked_app, count(*) AS n FROM bronze.products GROUP BY ALL ORDER BY n DESC"))

print("Silver parsed values:")
display(q("SELECT typeof(has_linked_app), has_linked_app, count(*) AS n FROM silver.dim_products GROUP BY ALL ORDER BY n DESC"))

Bronze raw values:


,typeof(has_linked_app),has_linked_app,n
0,VARCHAR,False,200142
1,VARCHAR,True,199858


Silver parsed values:


,typeof(has_linked_app),has_linked_app,n
0,BOOLEAN,False,200142
1,BOOLEAN,True,199858


## 12. Known issues recap

- **`opening_branch_id -> branches.branch_id`**: confirmed clean (0 orphans, 100% populated) in the earlier branch_id investigation -- section 6 re-derives this directly against your current data.
- No `customer_status`-style domain drift found here: `product_status`'s real values (`Active, Blocked, Closed, Suspended`) match both the dictionary and `contracts.py` exactly.

## Checklist

- [ ] Column inventory (1): spec/table agree, `current_balance_usd`/`credit_limit_usd` present with correct types
- [ ] Row reconciliation (2): no FX-join fan-out, no duplicate `product_id` in Silver
- [ ] NOT NULL audit (3): all 12 declared columns actually populated
- [ ] UNIQUE audit (4): `product_number` genuinely unique, or note any real collisions found
- [ ] Domain checks (5): note any `product_type`/`currency`/`opening_channel` value that looks like an error rather than a real category
- [ ] FK integrity (6): both `customer_id` and `opening_branch_id` at 0 orphans
- [ ] FX conversion (7): full currency coverage, exact USD passthrough, correct non-USD math
- [ ] Numeric ranges (8): no negative balances/limits/rates/DPD
- [ ] Date sanity (9): no expirations before opening, no transactions/updates before opening, no anomalous window violations beyond what's explained
- [ ] Cross-field consistency (10): credit_limit/days_past_due population lines up with product_type as expected, or note if it doesn't
- [ ] `has_linked_app` (11): NULL rate (if any) explained by genuinely unparseable source values

In [31]:
con.close()
print("OK")

OK
